# 🏥 MedPredict AI v2 — Training Pipeline

**End-to-end ML pipeline:** Data loading → Feature engineering → Model training → Evaluation → SHAP Explainability → Save artifacts

> ⚠️ Run cells top-to-bottom. All models and encoders will be saved to `../models/`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import shap

print("✅ All imports successful")
print(f"Pandas: {pd.__version__} | NumPy: {np.__version__}")


## 📂 Step 1: Load & Explore Dataset

In [ ]:
df = pd.read_csv("../dataset/diseases.csv")

print(f"Shape: {df.shape}")
print(f"Diseases: {df['disease'].nunique()}")
print(f"Samples per disease:\n{df['disease'].value_counts()}")
df.head()


## 📊 Step 2: Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Disease distribution
disease_counts = df['disease'].value_counts()
axes[0].barh(disease_counts.index, disease_counts.values, color='steelblue', edgecolor='white')
axes[0].set_xlabel("Sample Count")
axes[0].set_title("Disease Distribution", fontsize=14, fontweight='bold')
axes[0].invert_yaxis()

# Symptom frequency
symptom_cols = [c for c in df.columns if c != 'disease']
symptom_freq = df[symptom_cols].sum().sort_values(ascending=False)
axes[1].bar(range(len(symptom_freq[:20])), symptom_freq[:20].values, color='coral', edgecolor='white')
axes[1].set_xticks(range(20))
axes[1].set_xticklabels(symptom_freq[:20].index, rotation=45, ha='right', fontsize=8)
axes[1].set_title("Top 20 Most Common Symptoms", fontsize=14, fontweight='bold')
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("../dataset/eda_plots.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Total symptoms: {len(symptom_cols)}")


## 🔧 Step 3: Feature Engineering & Split

In [ ]:
symptom_cols = [c for c in df.columns if c != 'disease']
X = df[symptom_cols].values
y = df['disease'].values

# Encode target labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"Features (symptoms): {len(symptom_cols)}")
print(f"Classes (diseases): {len(le.classes_)}")
print(f"Class names: {list(le.classes_)}")

# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f"\nTrain samples: {X_train.shape[0]}")
print(f"Test samples:  {X_test.shape[0]}")


## 🌲 Step 4a: Train Random Forest

In [ ]:
print("Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=4,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)

rf_metrics = {
    "accuracy":  round(accuracy_score(y_test, rf_pred) * 100, 2),
    "precision": round(precision_score(y_test, rf_pred, average="weighted") * 100, 2),
    "recall":    round(recall_score(y_test, rf_pred, average="weighted") * 100, 2),
    "f1":        round(f1_score(y_test, rf_pred, average="weighted") * 100, 2),
}

print("\n📊 Random Forest Results:")
for k, v in rf_metrics.items():
    print(f"  {k.capitalize()}: {v}%")


## ⚡ Step 4b: Train XGBoost

In [ ]:
print("Training XGBoost...")
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)

xgb_metrics = {
    "accuracy":  round(accuracy_score(y_test, xgb_pred) * 100, 2),
    "precision": round(precision_score(y_test, xgb_pred, average="weighted") * 100, 2),
    "recall":    round(recall_score(y_test, xgb_pred, average="weighted") * 100, 2),
    "f1":        round(f1_score(y_test, xgb_pred, average="weighted") * 100, 2),
}

print("\n📊 XGBoost Results:")
for k, v in xgb_metrics.items():
    print(f"  {k.capitalize()}: {v}%")


## 📈 Step 5: Model Comparison

In [ ]:
metrics = ["accuracy", "precision", "recall", "f1"]
rf_vals  = [rf_metrics[m]  for m in metrics]
xgb_vals = [xgb_metrics[m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, rf_vals,  width, label="Random Forest", color="#1a73e8", alpha=0.85)
bars2 = ax.bar(x + width/2, xgb_vals, width, label="XGBoost",       color="#34a853", alpha=0.85)

ax.set_xlabel("Metric")
ax.set_ylabel("Score (%)")
ax.set_title("Model Comparison: Random Forest vs XGBoost", fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([m.capitalize() for m in metrics])
ax.set_ylim(85, 100)
ax.legend()
ax.bar_label(bars1, fmt="%.1f%%", padding=3, fontsize=9)
ax.bar_label(bars2, fmt="%.1f%%", padding=3, fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("../dataset/model_comparison.png", dpi=150, bbox_inches='tight')
plt.show()


## 🔲 Step 6: Confusion Matrix (Best Model)

In [ ]:
best_pred = rf_pred if rf_metrics["f1"] >= xgb_metrics["f1"] else xgb_pred
best_name = "Random Forest" if rf_metrics["f1"] >= xgb_metrics["f1"] else "XGBoost"

cm = confusion_matrix(y_test, best_pred)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=le.classes_, yticklabels=le.classes_,
    ax=ax, linewidths=0.5
)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("Actual", fontsize=12)
ax.set_title(f"Confusion Matrix — {best_name}", fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig("../dataset/confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()


## 🔍 Step 7: SHAP Explainability

In [ ]:
print("Computing SHAP values (this may take ~30 seconds)...")
explainer = shap.TreeExplainer(rf)
# Use a sample of test data for speed
sample_idx = np.random.choice(len(X_test), min(100, len(X_test)), replace=False)
shap_values = explainer.shap_values(X_test[sample_idx])

# Summary plot — overall feature importance
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_test[sample_idx],
    feature_names=symptom_cols,
    max_display=20,
    show=False,
    plot_type="bar"
)
plt.title("SHAP Feature Importance (Top 20 Symptoms)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("../dataset/shap_importance.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ SHAP computation complete")


## 💾 Step 8: Save All Artifacts

In [ ]:
import os
os.makedirs("../models", exist_ok=True)

# Pick best model
best_model = rf if rf_metrics["f1"] >= xgb_metrics["f1"] else xgb
best_name  = "RandomForest" if rf_metrics["f1"] >= xgb_metrics["f1"] else "XGBoost"

# Save
joblib.dump(best_model, "../models/model.pkl")
joblib.dump(rf,          "../models/rf_model.pkl")
joblib.dump(xgb,         "../models/xgb_model.pkl")
joblib.dump(le,          "../models/label_encoder.pkl")
joblib.dump(symptom_cols,"../models/symptoms.pkl")

stats = {
    "RandomForest": rf_metrics,
    "XGBoost":      xgb_metrics,
    "best_model":   best_name,
}
with open("../models/model_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print("✅ All artifacts saved to ../models/")
print(f"   Best model: {best_name}")
for fname in os.listdir("../models"):
    size = os.path.getsize(f"../models/{fname}")
    print(f"   {fname}: {size:,} bytes")


## 🧪 Step 9: Quick Inference Test

In [ ]:
# Test prediction
test_symptoms = ["fever", "cough", "headache", "fatigue", "chills"]

vector = [0] * len(symptom_cols)
for s in test_symptoms:
    if s in symptom_cols:
        vector[symptom_cols.index(s)] = 1

proba = best_model.predict_proba([vector])[0]
top3_idx = np.argsort(proba)[::-1][:3]

print(f"Input symptoms: {test_symptoms}")
print("\nTop 3 Predictions:")
for rank, idx in enumerate(top3_idx, 1):
    disease = le.inverse_transform([idx])[0]
    confidence = proba[idx] * 100
    print(f"  #{rank}: {disease} — {confidence:.1f}%")
